# Pipeline 5: Campaign Donation Amount Predictor

## 1. Problem Framing

### Business Problem
The organization runs fundraising campaigns (Year-End Hope, Back to School, Summer of Safety, GivingTuesday) but lacks insight into which donors to target for each campaign and how much to ask for. Currently, campaigns go out broadly with no personalization. Some donors receive asks that are too high (discouraging them) or too low (leaving money on the table). Staff have no way to set realistic campaign fundraising targets because they can't estimate expected yields.

A model that predicts the expected donation amount for a given donor in a given context enables three critical decisions: who to invite, how much to suggest, and what total to expect.

### Who Cares
- **Fundraising staff**: Need to personalize outreach — different donors respond to different campaigns at different giving levels.
- **Executive leadership**: Need realistic fundraising targets to budget for safehouse operations.
- **The girls**: Every optimized campaign dollar directly funds meals, education, counseling, and medical care.

### Why It Matters
The organization has no marketing team. Every donor interaction must be efficient and personalized. Predicting donation amounts helps staff focus limited time on the donors most likely to give meaningfully, ask for appropriate amounts, and forecast revenue for operational planning.

### Approach: Predictive AND Explanatory
- **Predictive goal**: Build a regression model that predicts the monetary donation amount (PHP) for a given donor-donation context. This powers a "Suggested Ask" feature on the Donors page and campaign revenue forecasting.
- **Explanatory goal**: Build an OLS regression to understand which donor characteristics and donation contexts drive higher giving. Is it the campaign? The channel? Recurring status? Relationship type? These coefficients directly inform campaign targeting strategy.

## 2. Data Acquisition, Preparation & Exploration

We join `donations` (monetary only) with `supporters` to create a donation-level dataset. Each row is a single monetary donation with both the donation context (campaign, channel, recurring) and the donor's profile (type, acquisition channel, relationship type).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.figsize'] = (12, 6)

# ----- Load data -----
DATA_DIR = "lighthouse_csv_v7"  # Adjust this path relative to your notebook location

supporters = pd.read_csv(f"{DATA_DIR}/supporters.csv")
donations = pd.read_csv(f"{DATA_DIR}/donations.csv")

# Filter to monetary donations
donations['donation_date'] = pd.to_datetime(donations['donation_date'])
donations['amount'] = pd.to_numeric(donations['amount'], errors='coerce')
monetary = donations[donations['donation_type'] == 'Monetary'].copy()

print(f"Total donations: {len(donations)}")
print(f"Monetary donations: {len(monetary)}")
print(f"Unique monetary donors: {monetary['supporter_id'].nunique()}")
print(f"Amount range: {monetary['amount'].min():.0f} - {monetary['amount'].max():.0f} PHP")

### Feature Engineering

We build features at the donation level (each row = one monetary gift) with both donor profile context and donation-specific context. We also engineer historical donor behavior features computed from prior donations.

In [ ]:
# Join with supporter profiles
df = monetary.merge(supporters[['supporter_id', 'supporter_type', 'relationship_type',
                                 'acquisition_channel', 'region', 'country', 'status']],
                     on='supporter_id', how='left')

# Campaign feature: fill missing campaign names
df['has_campaign'] = df['campaign_name'].notna().astype(int)
df['campaign_name'] = df['campaign_name'].fillna('No Campaign')

# Time features from donation date
df['donation_month'] = df['donation_date'].dt.month
df['donation_quarter'] = df['donation_date'].dt.quarter
df['donation_dow'] = df['donation_date'].dt.dayofweek
df['is_year_end'] = (df['donation_month'] >= 11).astype(int)

# Recurring flag as int
df['is_recurring'] = df['is_recurring'].astype(int)

# Donor history features (computed from all prior donations for each donor)
# Sort by date so we compute features using only past data
df = df.sort_values(['supporter_id', 'donation_date']).reset_index(drop=True)

# Cumulative donor statistics
donor_hist = []
for sid, group in df.groupby('supporter_id'):
    for i, (idx, row) in enumerate(group.iterrows()):
        prior = group.iloc[:i]
        donor_hist.append({
            'idx': idx,
            'prior_donation_count': len(prior),
            'prior_total_amount': prior['amount'].sum() if len(prior) > 0 else 0,
            'prior_avg_amount': prior['amount'].mean() if len(prior) > 0 else 0,
            'prior_max_amount': prior['amount'].max() if len(prior) > 0 else 0,
            'days_since_last': (row['donation_date'] - prior['donation_date'].max()).days if len(prior) > 0 else 0,
        })

hist_df = pd.DataFrame(donor_hist).set_index('idx')
df = df.join(hist_df)

# Log transform the target (right-skewed)
df['log_amount'] = np.log1p(df['amount'])

print(f"Donation-level dataset: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")

In [ ]:
# Univariate statistics
print("Donation Amount Statistics:")
print("=" * 50)
print(df['amount'].describe().round(2))
print(f"\nSkewness: {df['amount'].skew():.2f}")
print(f"Log-transformed skewness: {df['log_amount'].skew():.2f}")

print("\nDonor History Features:")
print(df[['prior_donation_count', 'prior_total_amount', 'prior_avg_amount', 'days_since_last']].describe().round(2))

print("\nMissing values:")
missing = df.isnull().sum()
missing = missing[missing > 0]
if len(missing) > 0:
    print(missing)
else:
    print("No missing values.")

In [ ]:
# Outlier analysis
Q1 = df['amount'].quantile(0.25)
Q3 = df['amount'].quantile(0.75)
IQR = Q3 - Q1

print(f"Outlier Analysis (IQR method):")
print(f"  Q1: {Q1:,.0f}, Q3: {Q3:,.0f}, IQR: {IQR:,.0f}")
print(f"  Upper bound (Q3 + 1.5*IQR): {Q3 + 1.5*IQR:,.0f}")
outliers = df[df['amount'] > Q3 + 1.5 * IQR]
print(f"  Outliers: {len(outliers)} ({len(outliers)/len(df):.1%})")
print(f"\nStrategy: Using log1p transformation to handle right-skew.")
print("Large donations are real and meaningful — they represent major gifts we want to predict, not remove.")

### Exploratory Analysis

In [ ]:
# Univariate distributions
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

axes[0,0].hist(df['amount'], bins=20, color='#1f77b4', edgecolor='white')
axes[0,0].set_title('Donation Amount Distribution (PHP)')

axes[0,1].hist(df['log_amount'], bins=20, color='#2ca02c', edgecolor='white')
axes[0,1].set_title('Log-Transformed Amount')

axes[0,2].hist(df['prior_donation_count'], bins=15, color='#ff7f0e', edgecolor='white')
axes[0,2].set_title('Prior Donation Count')

sns.barplot(data=df, x='campaign_name', y='amount', ax=axes[1,0], palette='viridis')
axes[1,0].set_title('Avg Amount by Campaign')
axes[1,0].tick_params(axis='x', rotation=45)

sns.barplot(data=df, x='channel_source', y='amount', ax=axes[1,1], palette='viridis')
axes[1,1].set_title('Avg Amount by Channel')
axes[1,1].tick_params(axis='x', rotation=45)

sns.barplot(data=df, x='is_recurring', y='amount', ax=axes[1,2], palette='Set2')
axes[1,2].set_title('Recurring vs One-Time')
axes[1,2].set_xticklabels(['One-Time', 'Recurring'])

plt.tight_layout()
plt.show()

In [ ]:
# Bivariate: amount by donor profile
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.barplot(data=df, x='acquisition_channel', y='amount', ax=axes[0], palette='viridis')
axes[0].set_title('Avg Amount by Acquisition Channel')
axes[0].tick_params(axis='x', rotation=45)

sns.barplot(data=df, x='relationship_type', y='amount', ax=axes[1], palette='viridis')
axes[1].set_title('Avg Amount by Relationship Type')

sns.scatterplot(data=df, x='prior_avg_amount', y='amount', hue='is_recurring', ax=axes[2], palette='Set2', alpha=0.6)
axes[2].set_title('Prior Avg Amount vs Current Amount')

plt.tight_layout()
plt.show()

In [ ]:
# Campaign performance summary
campaign_summary = df.groupby('campaign_name').agg(
    donations=('amount', 'count'),
    total_php=('amount', 'sum'),
    avg_php=('amount', 'mean'),
    median_php=('amount', 'median'),
    recurring_rate=('is_recurring', 'mean')
).round(2).sort_values('avg_php', ascending=False)

print("Campaign Performance Summary:")
print(campaign_summary.to_string())

In [ ]:
# Correlation matrix
numeric_cols = ['amount', 'is_recurring', 'has_campaign', 'prior_donation_count',
                'prior_total_amount', 'prior_avg_amount', 'prior_max_amount',
                'days_since_last', 'donation_month', 'is_year_end']
corr = df[numeric_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, square=True)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 3. Modeling & Feature Selection

We build an OLS model for explanation and tree-based regressors for prediction. Target is `log_amount` (log-transformed donation PHP). We use donation-level features (campaign, channel, recurring) plus donor profile features plus historical giving patterns.

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm
import joblib

# Define features
numeric_features = ['is_recurring', 'has_campaign', 'prior_donation_count',
                    'prior_total_amount', 'prior_avg_amount', 'prior_max_amount',
                    'days_since_last', 'donation_month', 'donation_quarter',
                    'is_year_end']
categorical_features = ['campaign_name', 'channel_source', 'acquisition_channel',
                        'relationship_type']

target = 'log_amount'

# Preprocessing
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), categorical_features)
])

X = df[numeric_features + categorical_features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape[0]}, Test: {X_test.shape[0]}")
print(f"Train mean amount: {np.expm1(y_train).mean():.0f} PHP")
print(f"Test mean amount: {np.expm1(y_test).mean():.0f} PHP")

### Explanatory Model: OLS Regression with VIF Check

In [ ]:
# Prepare OLS data
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

feature_names = numeric_features + list(
    preprocessor.transformers_[1][1].get_feature_names_out(categorical_features))

X_train_ols = pd.DataFrame(X_train_processed, columns=feature_names)
X_train_ols = sm.add_constant(X_train_ols)
X_test_ols = pd.DataFrame(X_test_processed, columns=feature_names)
X_test_ols = sm.add_constant(X_test_ols)

# Fit OLS
ols_model = sm.OLS(y_train.values, X_train_ols).fit()
print(ols_model.summary())

In [ ]:
# VIF check
vif_data = pd.DataFrame({
    'Feature': feature_names,
    'VIF': [variance_inflation_factor(X_train_ols.values, i+1) for i in range(len(feature_names))]
}).sort_values('VIF', ascending=False)

print("Variance Inflation Factors (VIF > 10 = problematic):")
print(vif_data.head(15).to_string(index=False))

high_vif = vif_data[vif_data['VIF'] > 10]
if len(high_vif) > 0:
    print(f"\n{len(high_vif)} features with VIF > 10.")
    print("For causal interpretation, these should be interpreted cautiously.")
else:
    print("\nNo severe multicollinearity detected.")

In [ ]:
# Significant coefficients visualization
coef_df = pd.DataFrame({
    'feature': ols_model.params.index,
    'coefficient': ols_model.params.values,
    'p_value': ols_model.pvalues.values
})
coef_df = coef_df[coef_df['feature'] != 'const']

significant = coef_df[coef_df['p_value'] < 0.1].sort_values('coefficient')

fig, ax = plt.subplots(figsize=(10, max(6, len(significant) * 0.35)))
colors = ['#2ca02c' if c > 0 else '#d62728' for c in significant['coefficient']]
ax.barh(significant['feature'], significant['coefficient'], color=colors)
ax.set_xlabel('OLS Coefficient (log-scale donation amount)')
ax.set_title('Significant Drivers of Donation Amount')
ax.axvline(x=0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

### Predictive Models with Hyperparameter Tuning

In [ ]:
# Compare models
models = {
    'OLS': SkPipeline([('prep', preprocessor), ('reg', LinearRegression())]),
    'Ridge': SkPipeline([('prep', preprocessor), ('reg', Ridge(alpha=1.0))]),
    'Decision Tree': SkPipeline([('prep', preprocessor), ('reg', DecisionTreeRegressor(random_state=42, max_depth=5))]),
    'Random Forest': SkPipeline([('prep', preprocessor), ('reg', RandomForestRegressor(random_state=42, n_estimators=100, max_depth=5))]),
    'Gradient Boosting': SkPipeline([('prep', preprocessor), ('reg', GradientBoostingRegressor(random_state=42, n_estimators=100, max_depth=4))]),
}

results = {}
for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_test_php = np.expm1(y_test)
    y_pred_php = np.expm1(y_pred)
    
    cv_scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring='r2')
    
    results[name] = {
        'r2': r2_score(y_test, y_pred),
        'rmse_php': np.sqrt(mean_squared_error(y_test_php, y_pred_php)),
        'mae_php': mean_absolute_error(y_test_php, y_pred_php),
        'cv_r2_mean': cv_scores.mean(),
        'cv_r2_std': cv_scores.std()
    }
    
    print(f"\n{name}")
    print(f"  CV R²: {cv_scores.mean():.3f} (+/- {cv_scores.std():.3f})")
    print(f"  Test R²: {r2_score(y_test, y_pred):.3f}")
    print(f"  RMSE (PHP): {results[name]['rmse_php']:,.0f}")
    print(f"  MAE (PHP):  {results[name]['mae_php']:,.0f}")

In [ ]:
# Hyperparameter tuning for Gradient Boosting
param_grid = {
    'reg__n_estimators': [100, 200, 300],
    'reg__max_depth': [3, 4, 5],
    'reg__learning_rate': [0.05, 0.1, 0.15]
}

gb_pipe = SkPipeline([('prep', preprocessor),
    ('reg', GradientBoostingRegressor(random_state=42))])

grid_search = GridSearchCV(gb_pipe, param_grid, cv=5, scoring='r2', n_jobs=-1)
grid_search.fit(X_train, y_train)

print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV R²: {grid_search.best_score_:.3f}")

best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test)
print(f"Test R²: {r2_score(y_test, y_pred_best):.3f}")
print(f"Test RMSE (PHP): {np.sqrt(mean_squared_error(np.expm1(y_test), np.expm1(y_pred_best))):,.0f}")

In [ ]:
# Feature importance
feat_imp = pd.DataFrame({
    'feature': feature_names,
    'importance': best_model.named_steps['reg'].feature_importances_
}).sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
sns.barplot(data=feat_imp.head(15), x='importance', y='feature', ax=ax, palette='viridis')
ax.set_title('Feature Importance: Donation Amount Prediction')
plt.tight_layout()
plt.show()

print("Top 15 features:")
print(feat_imp.head(15).to_string(index=False))

## 4. Evaluation & Interpretation

### Metrics
We evaluate using R² (explained variance), RMSE in PHP (interpretable error), and MAE in PHP. We also convert predictions to a classification view: did we predict the right giving tier?

### Business Interpretation
- **RMSE in PHP** tells staff: "The suggested ask amount will typically be within X PHP of the actual amount."
- **R²** tells leadership: "We can explain X% of the variation in donation amounts based on donor and campaign characteristics."
- **The coefficients** tell fundraising strategy: "Year-End Hope generates X% higher average gifts than direct appeals."

### Consequences of Errors
- **Overestimate** (suggest too high): Donor may feel pressured and not give at all. Moderate cost.
- **Underestimate** (suggest too low): Donor gives what's asked — less than they would have otherwise. Missed revenue.

Neither error is catastrophic, which makes this a lower-risk deployment than the reintegration or risk models. The suggestion is advisory and staff can adjust.

In [ ]:
# Actual vs predicted
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

y_test_php = np.expm1(y_test)
y_pred_php = np.expm1(y_pred_best)

axes[0].scatter(y_test_php, y_pred_php, alpha=0.5, color='#1f77b4')
max_val = max(y_test_php.max(), y_pred_php.max())
axes[0].plot([0, max_val], [0, max_val], 'r--', alpha=0.7)
axes[0].set_xlabel('Actual Donation (PHP)')
axes[0].set_ylabel('Predicted Donation (PHP)')
axes[0].set_title('Actual vs Predicted Donation Amount')

# Residuals
residuals = y_test_php - y_pred_php
axes[1].hist(residuals, bins=25, color='#2ca02c', edgecolor='white')
axes[1].set_xlabel('Residual (PHP)')
axes[1].set_title('Residual Distribution')
axes[1].axvline(x=0, color='red', linestyle='--')

plt.tight_layout()
plt.show()

In [ ]:
# Classification view: giving tiers
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

tier_bins = [0, 500, 1000, 2000, float('inf')]
tier_labels = ['<500', '500-1K', '1K-2K', '2K+']

y_test_tier = pd.cut(y_test_php, bins=tier_bins, labels=tier_labels)
y_pred_tier = pd.cut(y_pred_php.clip(0), bins=tier_bins, labels=tier_labels)

print("Giving Tier Classification:")
print(classification_report(y_test_tier, y_pred_tier, zero_division=0))

fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(y_test_tier, y_pred_tier,
    display_labels=tier_labels, cmap='Blues', ax=ax)
ax.set_title('Confusion Matrix: Giving Tier Prediction')
plt.tight_layout()
plt.show()

# Model comparison table
comparison = pd.DataFrame(results).T.round(3)
print("\nModel Comparison:")
print(comparison.to_string())

## 5. Causal and Relationship Analysis

### Key Findings

1. **Prior giving history is the strongest predictor**: Donors who have given more in the past (higher prior average, higher prior total) give more in the current donation. This is the single most powerful feature and has strong theoretical backing — past behavior predicts future behavior.

2. **Campaign context matters**: Named campaigns (Year-End Hope, Back to School, etc.) are associated with different giving levels compared to direct/no-campaign donations. Campaigns may create urgency, provide emotional connection, or attract different donor segments.

3. **Recurring donors give differently**: The recurring flag is associated with donation amounts, though the direction depends on context. Recurring donors may give smaller per-transaction amounts but more consistently, while one-time donors may give larger individual gifts.

4. **Acquisition channel influences giving**: How a donor was acquired (Website, SocialMedia, Event, WordOfMouth, PartnerReferral, Church) is associated with their giving level. Some channels attract higher-value donors.

5. **Relationship type (Local vs International)**: International donors and partner organizations may give at different levels than local individual donors, reflecting different economic contexts and organizational giving patterns.

6. **Seasonality**: Year-end donations tend to be higher, consistent with the "Year-End Hope" campaign and general charitable giving patterns.

### Causal Defensibility
- **Prior giving → current amount**: This is one of the most defensible predictive relationships (past behavior predicts future behavior) but it's not purely causal — it reflects stable donor characteristics (wealth, commitment) rather than a direct cause.
- **Campaign → amount**: Campaigns likely have a genuine causal effect on giving (they create urgency and emotional connection), but the effect is confounded by self-selection (donors who respond to campaigns may already be more engaged).
- **Acquisition channel**: These effects likely reflect donor segment characteristics, not a direct channel effect. A donor acquired through a church is a different person than one acquired through social media.
- **We cannot claim that switching a donor from "Direct" to "Campaign" will increase their giving.** But we can use these patterns to target the right donors with the right campaigns.

### Recommendations
1. **Use prior giving history as the primary input for "Suggested Ask"** — a donor who averages 2,000 PHP should see a higher suggested amount than one who averages 500 PHP.
2. **Tailor campaign invitations by donor profile** — different campaigns perform better with different donor segments.
3. **Set realistic campaign targets** by summing predicted amounts across invited donors.
4. **Track acquisition channel ROI** — invest in channels that produce higher-value donors long-term.

## 6. Deployment Notes

### How This Model Is Deployed
The trained model is serialized and served through a .NET API endpoint. For each donor, the backend computes their historical giving features from the donations table and passes them along with the campaign context to get a predicted amount.

### Web App Integration
- **Donors & Contributions page**: A "Suggested Ask" column appears next to each active donor, showing the predicted donation amount for the next campaign. Staff can filter by campaign to see predicted yields per campaign.
- **Campaign Planning Tool**: Staff select a campaign, and the model estimates total expected revenue based on the predicted amounts for all active donors. This helps set realistic fundraising targets.
- **Admin Dashboard**: A KPI card shows "Expected campaign yield: X PHP" for the active campaign.

### Production Scaling Note
With 234 monetary donations from 57 donors, the model has a reasonable training set for donation-level patterns. As the organization grows its donor base and runs more campaigns, the model retrains and predictions improve with more data.

### Model Export

In [ ]:
# Export model
joblib.dump(best_model, 'donation_amount_model.pkl')

model_config = {
    'numeric_features': numeric_features,
    'categorical_features': categorical_features,
    'target': 'log1p(amount)',
    'note': 'Exponentiate predictions with np.expm1() to get PHP values'
}
import json
with open('donation_amount_config.json', 'w') as f:
    json.dump(model_config, f, indent=2)
print("Model and config saved.")

In [ ]:
# Example: Predict suggested ask for different donor-campaign combinations
scenarios = pd.DataFrame([
    {'campaign_name': 'Year-End Hope', 'channel_source': 'Campaign', 'acquisition_channel': 'Website',
     'relationship_type': 'International', 'is_recurring': 1, 'has_campaign': 1,
     'prior_donation_count': 5, 'prior_total_amount': 8000, 'prior_avg_amount': 1600,
     'prior_max_amount': 3000, 'days_since_last': 45, 'donation_month': 12,
     'donation_quarter': 4, 'is_year_end': 1},
    {'campaign_name': 'Back to School', 'channel_source': 'SocialMedia', 'acquisition_channel': 'SocialMedia',
     'relationship_type': 'Local', 'is_recurring': 0, 'has_campaign': 1,
     'prior_donation_count': 1, 'prior_total_amount': 500, 'prior_avg_amount': 500,
     'prior_max_amount': 500, 'days_since_last': 180, 'donation_month': 6,
     'donation_quarter': 2, 'is_year_end': 0},
    {'campaign_name': 'No Campaign', 'channel_source': 'Direct', 'acquisition_channel': 'Church',
     'relationship_type': 'Local', 'is_recurring': 1, 'has_campaign': 0,
     'prior_donation_count': 10, 'prior_total_amount': 12000, 'prior_avg_amount': 1200,
     'prior_max_amount': 2500, 'days_since_last': 30, 'donation_month': 3,
     'donation_quarter': 1, 'is_year_end': 0},
])

pred_log = best_model.predict(scenarios)
pred_php = np.expm1(pred_log)

print("Suggested Ask Predictions:")
print("=" * 80)
for i, row in scenarios.iterrows():
    print(f"\nScenario {i+1}: {row['relationship_type']} donor | {row['campaign_name']} | "
          f"Prior avg: {row['prior_avg_amount']:,.0f} PHP | Recurring: {bool(row['is_recurring'])}")
    print(f"  Suggested ask: {pred_php[i]:,.0f} PHP")